# OpenQARP — Official Tutorial 02: Primitives (Colab Edition)

This notebook is the **third official tutorial** from the OpenQARP repository
(`examples/tutorial_02_primitives.ipynb`), reproduced as-is from
https://github.com/OpenQARP/openqarp/tree/main/examples and wrapped with a
Colab install cell so it runs standalone, with nothing else changed.

Tutorials 00–01 covered `Block` (what a circuit *does*). This one covers the
second layer — `PrimitiveAlgorithm`, which declares **what you want to
extract**: samples, overlaps, expectation values, or transition amplitudes,
exactly or shot-based, all through the same `bra`/`operator`/`ket` interface.

By Carlos Araque  
@KentryOps Data Labs  
Quantum researchers  
20-09-2026

## Setup (added for Colab — not part of the original tutorial)

Installs `openqarp` from PyPI. Wheels ship for Linux/macOS/Windows on Python
3.11–3.14, so this is a fast binary install on Colab's default runtime, no
compilation needed.

In [1]:
!pip install -q openqarp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 58.5 MB/s eta 0:00:00


In [2]:
import qarp

qarp.__version__

'0.1.0'

---

# Tutorial 02 — Primitives: from circuits to numbers

A `PrimitiveAlgorithm` declares **what you want to extract** from circuits.
You hand primitives to an engine; the engine returns one result per primitive.

The two workhorses:

| Primitive | Returns | How |
|---|---|---|
| `Sampler` | bitstring distribution (`SamplingDictionary`) | simulates and samples shots |
| `StateVector` | exact `float`/`complex` scalar | exact statevector contraction, no shots |

Shot-based *estimators* of the same scalars exist too (`TermwiseHadamardTest`,
`SWAPTest`, `PauliAveraging`, ...) — same interface, sampled instead of exact.
See `mwe_primitive_algorithms.ipynb` for the catalogue.

## 1. The target is inferred from what you pass

Primitives take up to three ingredients — `bra`, `operator`, `ket` — and infer
the quantity (their `target`) from which ones you supply:

* `ket` only → **sampling**
* `bra`, `ket` → **overlap** $\langle \mathrm{bra}|\mathrm{ket}\rangle$
* `bra=ket`, `operator` → **expectation value** $\langle \psi|H|\psi\rangle$
* `bra`, `operator`, `ket` → **transition amplitude** $\langle \mathrm{bra}|H|\mathrm{ket}\rangle$

In [3]:
import numpy as np
from qarp.operators import QubitOperator
from qarp.blocks import ComputationalBasisStateBlock, HEABlock, HnBlock
from qarp.algorithms import Sampler, StateVector

psi = HEABlock(3, 2, True, True, True, False).build()
psi_01 = psi.set_symbols({s: 0.1 for s in psi.symbols}).build()
plus = HnBlock(n_qubits=3).build()

H = QubitOperator("Z0 Z1", -1.0) + QubitOperator("X2", 0.5)

for prim in (
    Sampler(ket=psi_01),
    StateVector(bra=plus, ket=psi_01),
    StateVector(bra=psi_01, operator=H, ket=psi_01),
    StateVector(bra=plus, operator=H, ket=psi_01),
):
    prim.build()
    print(f"{type(prim).__name__:12s} -> {prim.target}")

Sampler      -> SAMPLING
StateVector  -> OVERLAP
StateVector  -> EXPECTATION_VALUE
StateVector  -> TRANSITION_AMPLITUDE


## 2. Exact expectation values with `StateVector`

In [4]:
from qarp.engines import QarpEngine

expval = StateVector(bra=psi, operator=H, ket=psi)   # symbolic ansatz: values at run time

engine = QarpEngine()
engine.build([expval])

values = dict(zip(psi.symbols, np.linspace(0.0, 1.0, len(psi.symbols))))
energy = engine.run(values)[0]
print("<psi|H|psi> =", energy)
print("stored on the primitive too:", expval.result)

<psi|H|psi> = (-0.3592226466145432+0j)
stored on the primitive too: (-0.3592226466145432+0j)


Notes:

* The operator is a `qarp.operators.QubitOperator` (openfermion-compatible API);
  its qubit indices are qarpx qubit indices, and `op.sparse_matrix()` realizes it in
  the same LSB convention as every qarpx statevector — no conversion anywhere.
* Only when you bring in an *external* MSB-first matrix or statevector (openfermion,
  cirq, pennylane) do you convert, with `qarp.endianness.msb_to_lsb_matrix` /
  `..._statevector` (eigenvalues need no conversion — they don't depend on bit order).

## 3. Overlaps

In [5]:
overlap = StateVector(bra=plus, ket=psi_01)
engine = QarpEngine()
engine.build([overlap])
print("<+++|psi(0.1)> =", engine.run()[0])

<+++|psi(0.1)> = (0.45732375474796383+0j)


## 4. Sampling, and shot-based estimation

`Sampler` returns the measured distribution; `n_shots` set on the primitive
overrides the engine default. For a *shot-based estimate of an expectation
value* (what real hardware gives you), use `TermwiseHadamardTest` or
`PauliAveraging` — same `bra/operator/ket` interface as `StateVector`:

In [6]:
from qarp.algorithms import TermwiseHadamardTest

exact = StateVector(bra=psi_01, operator=H, ket=psi_01)
estimated = TermwiseHadamardTest(bra=psi_01, operator=H, ket=psi_01, n_shots=20_000)

engine = QarpEngine(seed=7)
engine.build([exact, estimated])      # one engine, several primitives
res = engine.run()
print("exact:    ", res[0])
print("estimated:", res[1])

exact:     (-0.926547548611303+0j)
estimated: (-0.92945+0.010400000000000076j)


## Choosing between them

* **Developing / debugging an algorithm** → `StateVector`. Deterministic, fast,
  differentiable (tutorial 03 shows gradients).
* **Modelling what hardware would measure** → `Sampler` / Hadamard-test family
  with a finite `n_shots` and a seeded engine.

## Poke at it

* `prim.target`, `prim.n_shots`, `prim.result` (and `result_list` on termwise
  estimators — one entry per Pauli term)
* `prim.compiled_circuits` after `engine.build` — the transpiled command
  streams the engine will actually simulate

**Next:** tutorial_03_engines — the compile-once / run-many split, seeds,
batching, and gradients.

---

*Source: [OpenQARP/openqarp, examples/tutorial_02_primitives.ipynb](https://github.com/OpenQARP/openqarp/tree/main/examples), Apache License 2.0, Fujitsu Research of Europe.*